In [4]:
import pandas as pd

# El '..' sirve para retroceder a la carpeta principal y entrar a 'data'
ruta_archivo = '../data/DDoS-Friday-no-metadata.parquet'

# Usamos read_parquet en lugar de read_csv
df = pd.read_parquet(ruta_archivo)

# Mostramos el tamaño del archivo (filas, columnas) y las primeras 5 líneas
print(f"Tamaño del dataset: {df.shape}")
df.head()

Tamaño del dataset: (221264, 78)


,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,6,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
1,6,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
2,6,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
3,6,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
4,6,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign


In [6]:
# 1. Limpiar los nombres de las columnas (quitar espacios extra)
df.columns = df.columns.str.strip()

# 2. Ver cuántos tipos de tráfico tenemos en este archivo
print("--- Distribución de las etiquetas (Tráfico normal vs Ataques) ---")
# Nota: A veces la columna se llama 'Label' o 'label', ajustamos por si acaso
columna_label = 'Label' if 'Label' in df.columns else 'label'
print(df[columna_label].value_counts())

# 3. Revisar si hay valores nulos (NaN)
total_nulos = df.isnull().sum().sum()
print(f"\nTotal de valores nulos en el dataset: {total_nulos}")

--- Distribución de las etiquetas (Tráfico normal vs Ataques) ---
Label
DDoS      128014
Benign     93250
Name: count, dtype: int64

Total de valores nulos en el dataset: 0


In [7]:
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Limpiar valores infinitos (muy comunes en datasets de red)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# 2. Convertir etiquetas a formato binario para el modelo
# 0 = Tráfico Normal, 1 = Ataque
df['Label'] = df['Label'].map({'Benign': 0, 'DDoS': 1})

# 3. Separar las características (X) de la etiqueta a predecir (y)
X = df.drop(columns=['Label'])
y = df['Label']

# 4. Dividir en conjunto de entrenamiento (70%) y validación (30%)
# 'stratify=y' asegura que mantengamos la misma proporción de ataques en ambas partes
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Flujos para entrenar la IA: {X_train.shape[0]}")
print(f"Flujos para probar la IA: {X_test.shape[0]}")

Flujos para entrenar la IA: 154884
Flujos para probar la IA: 66380


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1. Inicializar el modelo Random Forest
# Usamos n_jobs=-1 para usar todos los núcleos de tu procesador y acelerar el entrenamiento
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# 2. Entrenar el modelo con los datos de entrenamiento
print("Entrenando el modelo... (esto puede tomar unos segundos)")
modelo_rf.fit(X_train, y_train)

# 3. Hacer predicciones con los datos de prueba (los que la IA no ha visto antes)
predicciones = modelo_rf.predict(X_test)

# 4. Evaluar los resultados
print("\n--- Reporte de Clasificación ---")
print(classification_report(y_test, predicciones, target_names=['Normal (0)', 'Ataque DDoS (1)']))

print("--- Matriz de Confusión ---")
print(confusion_matrix(y_test, predicciones))

Entrenando el modelo... (esto puede tomar unos segundos)

--- Reporte de Clasificación ---
                 precision    recall  f1-score   support

     Normal (0)       1.00      1.00      1.00     27975
Ataque DDoS (1)       1.00      1.00      1.00     38405

       accuracy                           1.00     66380
      macro avg       1.00      1.00      1.00     66380
   weighted avg       1.00      1.00      1.00     66380

--- Matriz de Confusión ---
[[27973     2]
 [   10 38395]]


In [9]:
import joblib
import os

# Aseguramos que la carpeta de modelos exista
os.makedirs('../models', exist_ok=True)

# Ruta donde guardaremos el archivo
ruta_modelo = '../models/nids_random_forest.joblib'

# Guardamos el modelo entrenado
joblib.dump(modelo_rf, ruta_modelo)

print(f"¡Modelo guardado exitosamente en: {ruta_modelo}!")

¡Modelo guardado exitosamente en: ../models/nids_random_forest.joblib!
